# EDA & Visualization — Reading Data Before Modeling

## The Purpose of EDA

You can build a model in 5 lines of sklearn.
That model will be wrong in ways that are very hard to diagnose — unless you did EDA first.

EDA answers the questions that determine every modeling decision:

| Question | Drives |
|---|---|
| What's the target distribution? | Whether to log-transform it |
| Which features correlate with target? | Where to spend feature engineering effort |
| Which features correlate with each other? | Multicollinearity concerns, redundancy |
| Are there outliers? | Whether to winsorize, remove, or flag |
| How are categories distributed? | Whether groupby features will be stable |
| What patterns exist visually? | Non-linearities, interactions, thresholds |

## This Notebook

We run a full EDA on a House Prices dataset — the same structure as the Kaggle competition.

| Section | What You'll Build |
|---|---|
| 1 | Univariate: distribution of every numeric feature |
| 2 | Target deep-dive: why log-transform SalePrice |
| 3 | Bivariate: feature vs target (scatter, box, violin) |
| 4 | Correlation: heatmap + ranked feature importance |
| 5 | Categorical: grouped distributions |
| 6 | Pairwise: multi-feature relationships |
| 7 | The EDA report: decisions for cleaning & engineering |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats as scipy_stats
from pathlib import Path

np.random.seed(42)
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 110)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=0.95)

# ── Synthetic House Prices dataset (Kaggle-style) ───────────────────────────
n = 600

neighborhoods = ['NAmes','CollgCr','OldTown','Edwards','Somerst',
                 'NridgHt','Gilbert','Sawyer','BrkSide','Crawfor']
bldg_types    = ['1Fam','TwnhsE','Duplex','2fmCon']
conditions    = ['Excellent','Good','Typical','Fair','Poor']

qual    = np.random.randint(3, 11, n)
area    = np.random.normal(1500, 420, n).clip(400, 5000).round(0)
yr_built = np.random.randint(1930, 2023, n)
garage  = np.random.choice([0,1,2,3], n, p=[0.05,0.18,0.62,0.15]).astype(float)
bsmt    = np.random.normal(950, 320, n).clip(0, 3000).round(0)
lot     = np.random.lognormal(9.0, 0.5, n).clip(1500, 50000).round(0)
rooms   = np.random.randint(4, 12, n)
full_bath = np.random.choice([1,2,3], n, p=[0.30,0.60,0.10])
nbhd    = np.random.choice(neighborhoods, n,
          p=[0.13,0.13,0.09,0.09,0.11,0.11,0.11,0.09,0.07,0.07])
btype   = np.random.choice(bldg_types, n, p=[0.72,0.16,0.06,0.06])
cond    = np.random.choice(conditions, n, p=[0.10,0.30,0.40,0.15,0.05])

nbhd_premium = {'NAmes':0,'CollgCr':5000,'OldTown':-15000,'Edwards':-20000,
                'Somerst':15000,'NridgHt':40000,'Gilbert':10000,'Sawyer':-5000,
                'BrkSide':-25000,'Crawfor':8000}
cond_mult = {'Excellent':1.15,'Good':1.05,'Typical':1.0,'Fair':0.93,'Poor':0.85}

price = (
    area * 82 +
    qual * 9500 +
    (yr_built - 1930) * 420 +
    garage * 5500 +
    bsmt * 28 +
    np.log1p(lot) * 2500 +
    rooms * 1800 +
    full_bath * 7000 +
    np.array([nbhd_premium[x] for x in nbhd]) +
    np.random.normal(0, 18000, n)
)
price = price * np.array([cond_mult[x] for x in cond])
price = price.clip(55000, 780000).round(-3)

# Inject realistic missing values
miss_idx = np.random.permutation(n)
garage_na = miss_idx[:35]
bsmt_na   = miss_idx[35:55]

df = pd.DataFrame({
    'Id':          np.arange(1, n+1),
    'GrLivArea':   area,
    'OverallQual': qual,
    'YearBuilt':   yr_built,
    'GarageCars':  garage,
    'TotalBsmtSF': bsmt,
    'LotArea':     lot,
    'TotRmsAbvGrd':rooms,
    'FullBath':    full_bath,
    'Neighborhood':nbhd,
    'BldgType':    btype,
    'Condition':   cond,
    'SalePrice':   price,
})
df.loc[garage_na, 'GarageCars'] = np.nan
df.loc[bsmt_na,   'TotalBsmtSF'] = np.nan

# Clean for use (impute, don't need missing in most sections)
df_clean = df.copy()
df_clean['GarageCars'] = df_clean['GarageCars'].fillna(0)
df_clean['TotalBsmtSF'] = df_clean['TotalBsmtSF'].fillna(0)
df_clean['HouseAge']    = 2024 - df_clean['YearBuilt']
df_clean['LogPrice']    = np.log1p(df_clean['SalePrice'])
df_clean['log_SalePrice'] = df_clean['LogPrice']

NUMERIC_COLS = ['GrLivArea','OverallQual','YearBuilt','GarageCars',
                'TotalBsmtSF','LotArea','TotRmsAbvGrd','FullBath','SalePrice']
CAT_COLS     = ['Neighborhood','BldgType','Condition']
TARGET       = 'SalePrice'

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} cols")
print(f"Numeric: {len(NUMERIC_COLS)}   Categorical: {len(CAT_COLS)}   Target: {TARGET}")
print(f"Price range: ${df_clean[TARGET].min():,.0f} - ${df_clean[TARGET].max():,.0f}")


---
## Section 1 — Univariate Analysis: Understanding Each Feature Alone

**First question:** What does each feature look like in isolation?

For numeric features you want to know:
- Shape: symmetric, right-skewed, left-skewed, bimodal?
- Range: any impossible values (negative area, year=9999)?
- Outliers: long tail? extreme values?
- Zeros: zero-inflated? (many zeros + a tail = two different distributions)

**Tools:** histogram, boxplot, KDE, descriptive stats

**Decision output:** which features need log-transform, winsorization, or special handling


In [ ]:
print("=== Univariate Analysis ===")
print()

# Numeric summary with skewness added
skew_stats = df_clean[NUMERIC_COLS].apply(lambda c: c.skew()).round(2)
kurtosis   = df_clean[NUMERIC_COLS].apply(lambda c: c.kurtosis()).round(2)
miss_pct   = (df[NUMERIC_COLS].isnull().sum() / len(df) * 100).round(1)

summary = df_clean[NUMERIC_COLS].describe().T
summary['skewness']  = skew_stats
summary['kurtosis']  = kurtosis
summary['missing_%'] = miss_pct
print(summary[['count','mean','std','min','50%','max','skewness','missing_%']].round(1))
print()
print("Skewness interpretation:")
print("  |skew| < 0.5  : roughly symmetric -> ok as-is")
print("  0.5-1.0       : moderate skew -> consider transform")
print("  > 1.0         : high skew -> log/sqrt transform recommended")
print()

# Dashboard: histograms for all numeric features
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
colors = sns.color_palette("muted", len(NUMERIC_COLS))

for i, col in enumerate(NUMERIC_COLS):
    ax   = axes[i]
    data = df_clean[col].dropna()
    skew = data.skew()
    ax.hist(data, bins=35, color=colors[i], edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.set_title(col + "  (skew=" + f"{skew:+.2f})", fontsize=9, fontweight='bold')
    ax.set_xlabel(col, fontsize=8)
    ax.set_ylabel('count', fontsize=8)
    ax.tick_params(labelsize=7)
    # Add skew severity indicator
    if abs(skew) > 1:
        ax.set_facecolor('#fff5f5')
        ax.set_title(col + "  (skew=" + f"{skew:+.2f}) *", fontsize=9, fontweight='bold', color='darkred')

plt.suptitle("Univariate Distributions  (* = high skew, consider log-transform)",
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/tmp/eda_univariate.png', dpi=100, bbox_inches='tight')
plt.show()
print()
print("Highlighted in red: LotArea and SalePrice both have skew > 1")
print("Action: log-transform these before feeding to linear models")


---
## Section 2 — Target Deep-Dive: SalePrice

The target variable deserves its own section.

**Why:** Linear regression assumes **normally distributed residuals**.
If the target is right-skewed, a linear model:
- Over-predicts cheap houses (pulls toward the mean)
- Under-predicts expensive houses (can't reach the tail)
- Produces higher percentage errors on cheap houses

**The fix:** predict `log(SalePrice + 1)` instead of `SalePrice`.

The log transform compresses the right tail, making the distribution
more symmetric. After prediction, you exponentiate back: `np.expm1(pred)`.

This is standard practice in price prediction competitions.


In [ ]:
print("=== Target Variable: SalePrice ===")
print()

price     = df_clean['SalePrice']
log_price = np.log1p(price)

stats_raw = {'mean': price.mean(), 'median': price.median(),
             'std':  price.std(),  'skew':   price.skew()}
stats_log = {'mean': log_price.mean(), 'median': log_price.median(),
             'std':  log_price.std(),  'skew':   log_price.skew()}

print("SalePrice statistics:")
for k in ['mean','median','std','skew']:
    print(f"  {k:8}: raw={stats_raw[k]:>12,.0f}   log={stats_log[k]:.4f}")
print()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle("SalePrice: Raw vs Log Transform", fontsize=13, fontweight='bold')

# Row 1: raw SalePrice
axes[0,0].hist(price, bins=45, color='coral', edgecolor='white', linewidth=0.4)
axes[0,0].axvline(price.mean(),   color='navy',    linestyle='--', lw=1.5, label='mean')
axes[0,0].axvline(price.median(), color='darkgreen',linestyle='-',  lw=1.5, label='median')
axes[0,0].set_title("SalePrice  (skew=" + f"{price.skew():.2f})", fontweight='bold')
axes[0,0].set_xlabel('price ($)')
axes[0,0].legend(fontsize=8)

axes[0,1].boxplot(price.values, vert=True, patch_artist=True,
                   boxprops=dict(facecolor='coral', alpha=0.6),
                   medianprops=dict(color='darkred', linewidth=2))
axes[0,1].set_title("SalePrice boxplot")
axes[0,1].set_ylabel('price ($)')

scipy_stats.probplot(price.values, dist='norm', plot=axes[0,2])
axes[0,2].set_title("Q-Q plot (raw)\n-> deviates from Normal at tails")

# Row 2: log(SalePrice)
axes[1,0].hist(log_price, bins=45, color='steelblue', edgecolor='white', linewidth=0.4)
axes[1,0].axvline(log_price.mean(),   color='navy',    linestyle='--', lw=1.5, label='mean')
axes[1,0].axvline(log_price.median(), color='darkgreen',linestyle='-',  lw=1.5, label='median')
axes[1,0].set_title("log1p(SalePrice)  (skew=" + f"{log_price.skew():.2f})", fontweight='bold')
axes[1,0].set_xlabel('log(price + 1)')
axes[1,0].legend(fontsize=8)

axes[1,1].boxplot(log_price.values, vert=True, patch_artist=True,
                   boxprops=dict(facecolor='steelblue', alpha=0.6),
                   medianprops=dict(color='darkblue', linewidth=2))
axes[1,1].set_title("log(SalePrice) boxplot")
axes[1,1].set_ylabel('log(price + 1)')

scipy_stats.probplot(log_price.values, dist='norm', plot=axes[1,2])
axes[1,2].set_title("Q-Q plot (log-transformed)\n-> much closer to Normal")

plt.tight_layout()
plt.savefig('/tmp/eda_target.png', dpi=100, bbox_inches='tight')
plt.show()
print()
print("Conclusion: always predict log(SalePrice) for house price models.")
print("  model.predict(X) -> log predictions")
print("  np.expm1(predictions) -> back to dollars")


---
## Section 3 — Bivariate Analysis: Feature vs Target

**Second question:** How does each feature relate to SalePrice?

For numeric features:
- **Scatter plot** — direction, strength, linearity, outliers
- **Correlation coefficient** — linear relationship strength
- Does the relationship look linear? Or does it curve? (needs polynomial feature)

For categorical features:
- **Box plot** / **Violin plot** grouped by category
- Does price significantly differ across categories? (worth encoding this feature)

**What you're looking for:**
- Strong linear correlations -> keep as-is
- Curved relationships -> add polynomial term
- Weak correlations -> still include, tree models can use non-linear patterns
- Categorical with big spread -> important to encode well


In [ ]:
print("=== Bivariate: Numeric Features vs SalePrice ===")
print()

# Correlation with target
corrs = df_clean[NUMERIC_COLS].corr()['SalePrice'].drop('SalePrice').sort_values(key=abs, ascending=False)
print("Pearson correlation with SalePrice:")
for feat, corr in corrs.items():
    bar = '|' * int(abs(corr) * 25)
    sign = '+' if corr > 0 else '-'
    print(f"  {feat:15}: {corr:+.3f}  {sign}{bar}")
print()

# Scatter grid: top 6 features vs log(SalePrice)
top6 = corrs.abs().nlargest(6).index.tolist()
log_price = df_clean['LogPrice']

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for i, feat in enumerate(top6):
    ax   = axes[i]
    x    = df_clean[feat]
    corr = df_clean[[feat,'LogPrice']].corr().iloc[0,1]

    ax.scatter(x, log_price, alpha=0.3, s=12, color='steelblue', rasterized=True)

    # Regression line
    m, b, r, p, se = scipy_stats.linregress(x.dropna(), log_price[x.notna()])
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m * x_line + b, color='red', linewidth=1.5, label=f'r={corr:.3f}')

    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('log(SalePrice)', fontsize=9)
    ax.set_title(feat + "  vs  log(SalePrice)", fontsize=9, fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle("Top 6 Features vs log(SalePrice)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/eda_bivariate_num.png', dpi=100, bbox_inches='tight')
plt.show()

print()
print("=== Bivariate: Categorical Features vs SalePrice ===")
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for i, cat_col in enumerate(CAT_COLS):
    ax = axes[i]
    # Sort categories by median SalePrice for readability
    order = (df_clean.groupby(cat_col)['SalePrice']
             .median()
             .sort_values(ascending=False)
             .index.tolist())

    # Violin + strip (shows distribution shape AND individual points)
    sns.violinplot(data=df_clean, x=cat_col, y='SalePrice', order=order,
                   palette='muted', inner='quartile', ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=7)
    ax.set_title(cat_col + " vs SalePrice", fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

plt.suptitle("Categorical Features vs SalePrice", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/eda_bivariate_cat.png', dpi=100, bbox_inches='tight')
plt.show()
print()
print("Observations from violin plots:")
print("  Neighborhood: HUGE spread - NridgHt median ~$80k above OldTown/Edwards")
print("  BldgType: 1Fam tends higher; Duplex lower")
print("  Condition: ordinal relationship - Excellent >> Good > Typical > Poor")


---
## Section 4 — Correlation Analysis

### Two Goals

1. **Feature-target correlation** — which features are most predictive?
2. **Feature-feature correlation** — which features are redundant?

### Multicollinearity

When two features are highly correlated with each other (|r| > 0.8), they carry
redundant information. This causes problems for linear models:

- Coefficient estimates become unstable
- Standard errors inflate
- Hard to interpret which feature is "doing the work"

**Actions:**
- Keep one, drop the other — OR
- Use Ridge/Lasso which handle multicollinearity via regularization — OR
- Create an interaction/ratio feature that captures the combined signal

Tree models (RF, XGBoost) are unaffected by multicollinearity.


In [ ]:
print("=== Correlation Matrix ===")
print()

corr_matrix = df_clean[NUMERIC_COLS].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Full correlation heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)   # upper triangle only
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
            annot_kws={'size': 7}, ax=axes[0])
axes[0].set_title("Feature Correlation Matrix", fontsize=11, fontweight='bold')
axes[0].tick_params(labelsize=8)

# Ranked correlations with target
target_corrs = corr_matrix['SalePrice'].drop('SalePrice').sort_values()
colors = ['coral' if c < 0 else 'steelblue' for c in target_corrs]
axes[1].barh(target_corrs.index, target_corrs.values, color=colors, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title("Pearson r with SalePrice", fontsize=11, fontweight='bold')
axes[1].set_xlabel('correlation coefficient')
for i, (feat, val) in enumerate(target_corrs.items()):
    axes[1].text(val + (0.01 if val >= 0 else -0.01), i,
                 f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right',
                 fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/eda_correlation.png', dpi=100, bbox_inches='tight')
plt.show()
print()

# Find highly correlated feature pairs (multicollinearity check)
print("Highly correlated feature pairs (|r| > 0.60, excluding target):")
feats_only = [c for c in NUMERIC_COLS if c != 'SalePrice']
corr_feats = df_clean[feats_only].corr()
pairs = []
for i in range(len(feats_only)):
    for j in range(i+1, len(feats_only)):
        r = corr_feats.iloc[i, j]
        if abs(r) > 0.60:
            pairs.append((feats_only[i], feats_only[j], r))

pairs.sort(key=lambda x: abs(x[2]), reverse=True)
if pairs:
    for f1, f2, r in pairs:
        concern = " <- multicollinearity risk" if abs(r) > 0.8 else ""
        print(f"  {f1:15} x {f2:15}: r={r:+.3f}{concern}")
else:
    print("  None found above threshold")
print()
print("Strategy for correlated features:")
print("  - Using Ridge/Lasso? They handle it via regularization - keep both")
print("  - Need interpretability? Drop the weaker one")
print("  - Suspicious pair (area x rooms)? Maybe create area/rooms = avg room size")


---
## Section 5 — Categorical Deep Dives

### What to Look for in Categoricals

1. **Cardinality** — how many unique values? High cardinality (>20) needs careful encoding.
2. **Balance** — is one category 95% of rows? That category adds little signal.
3. **Price spread** — high between-group variance = informative feature
4. **Small groups** — categories with <5 samples are unstable and may overfit

### The Signal Test for Categoricals

A useful heuristic: plot the category means vs overall mean.
If category means vary a lot (large between-group variance relative to within-group),
the categorical feature is highly informative.

This is the intuition behind **ANOVA** and **target encoding**.


In [ ]:
print("=== Categorical Analysis ===")
print()

for cat_col in CAT_COLS:
    vc = df_clean[cat_col].value_counts()
    print(f"{cat_col}:  {df_clean[cat_col].nunique()} categories")
    group_means = df_clean.groupby(cat_col)['SalePrice'].agg(['mean','std','count'])
    group_means = group_means.sort_values('mean', ascending=False)
    overall     = df_clean['SalePrice'].mean()
    max_delta   = (group_means['mean'].max() - group_means['mean'].min()) / overall * 100
    print(f"  Overall mean: ${overall:,.0f}   Max category spread: {max_delta:.1f}% of overall mean")
    print(group_means.rename(columns={'mean':'mean_price','std':'std_price'}).round(0).head(5).to_string())
    print()

# Neighborhood deep dive (highest cardinality, most informative)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

nbhd_stats = (df_clean.groupby('Neighborhood')['SalePrice']
              .agg(['median','mean','std','count'])
              .sort_values('median', ascending=True))
overall_med = df_clean['SalePrice'].median()

# Horizontal bar: median price per neighborhood
ax = axes[0]
colors = ['steelblue' if m > overall_med else 'coral' for m in nbhd_stats['median']]
ax.barh(nbhd_stats.index, nbhd_stats['median'], color=colors, edgecolor='white')
ax.axvline(overall_med, color='black', linestyle='--', lw=1.5, label=f'overall median ${overall_med/1000:.0f}k')
ax.set_xlabel('median SalePrice ($)')
ax.set_title("Neighborhood Median Price", fontweight='bold')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.legend(fontsize=8)
# Add count annotations
for i, (nbhd, row) in enumerate(nbhd_stats.iterrows()):
    ax.text(row['median'] + 1000, i, f"n={int(row['count'])}", va='center', fontsize=7)

# Box plots with sample counts
ax = axes[1]
order = nbhd_stats.sort_values('median', ascending=False).index.tolist()
sns.boxplot(data=df_clean, x='Neighborhood', y='SalePrice', order=order,
            palette='muted', ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
ax.set_title("Neighborhood Price Distributions", fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

plt.suptitle("Neighborhood Analysis", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/eda_categorical.png', dpi=100, bbox_inches='tight')
plt.show()
print()
print("Key findings:")
print("  NridgHt is the premium neighborhood - median ~40-50% above average")
print("  OldTown, BrkSide, Edwards are discount neighborhoods")
print("  -> Neighborhood is one of the most powerful features to encode carefully")
print("  -> Consider target encoding instead of one-hot (10 categories = 10 dummy cols)")


---
## Section 6 — Pairplot & Multi-Feature Relationships

### Seeing Multiple Features at Once

A **pairplot** shows every pairwise relationship in one grid.
Each cell is a scatter plot of feature i vs feature j.
Diagonal shows the distribution of each feature.

Use it to spot:
- Non-linear relationships (curves in scatter plots)
- Outlier clusters
- Strong correlations you might have missed
- Bimodal features (two bumps on the diagonal)

**Practical note:** Only use on your top 5-6 features — beyond that it becomes unreadable.


In [ ]:
print("=== Pairplot: Top Features ===")
print()

# Select top 5 features by correlation + target
top5_feats = ['GrLivArea','OverallQual','YearBuilt','TotalBsmtSF','GarageCars','SalePrice']
df_pair    = df_clean[top5_feats].copy()

# Color by OverallQual (binned into low/mid/high)
df_pair['QualGroup'] = pd.cut(df_pair['OverallQual'], bins=[0,5,7,10],
                               labels=['Low(1-5)','Mid(6-7)','High(8-10)'])

g = sns.pairplot(df_pair, vars=top5_feats, hue='QualGroup',
                 palette={'Low(1-5)':'coral','Mid(6-7)':'gold','High(8-10)':'steelblue'},
                 diag_kind='kde', plot_kws={'alpha':0.35, 's':15},
                 corner=True)
g.fig.suptitle("Pairplot: Top Features colored by OverallQual",
               y=1.01, fontsize=12, fontweight='bold')
plt.savefig('/tmp/eda_pairplot.png', dpi=90, bbox_inches='tight')
plt.show()
print()

# Non-linearity check: OverallQual vs SalePrice
print("=== Non-linearity: OverallQual vs SalePrice ===")
print()
qual_stats = (df_clean.groupby('OverallQual')['SalePrice']
              .agg(['mean','median','count'])
              .round(0))
print(qual_stats)
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Mean price per quality grade
ax = axes[0]
ax.plot(qual_stats.index, qual_stats['mean']/1000, 'o-', color='steelblue', lw=2, ms=8)
ax.fill_between(qual_stats.index,
                (qual_stats['mean'] - qual_stats['mean'].std()/2)/1000,
                (qual_stats['mean'] + qual_stats['mean'].std()/2)/1000,
                alpha=0.2, color='steelblue')
ax.set_xlabel('OverallQual (1-10)')
ax.set_ylabel('mean SalePrice ($k)')
ax.set_title("Quality vs Mean Price\n(Near-linear but accelerating)")
ax.grid(True, alpha=0.4)

# YearBuilt vs price (running median)
ax = axes[1]
df_sorted = df_clean.sort_values('YearBuilt')
window    = df_sorted['SalePrice'].rolling(window=30, center=True).median()
ax.scatter(df_clean['YearBuilt'], df_clean['SalePrice']/1000,
           alpha=0.2, s=10, color='coral', label='individual')
ax.plot(df_sorted['YearBuilt'], window/1000, color='navy', lw=2, label='rolling median (n=30)')
ax.set_xlabel('YearBuilt')
ax.set_ylabel('SalePrice ($k)')
ax.set_title("YearBuilt vs SalePrice\n(Non-linear: pre-1970 homes lower)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/eda_nonlinear.png', dpi=100, bbox_inches='tight')
plt.show()
print()
print("Non-linearity findings:")
print("  OverallQual: price acceleration is near-exponential from 8->10")
print("  -> Consider OverallQual^2 or log(OverallQual)")
print("  YearBuilt: recent homes (2000+) significantly premium")
print("  -> HouseAge = 2024 - YearBuilt captures this; consider HouseAge^2")


---
## Section 7 — The EDA Report: Turning Findings into Decisions

### From Observations to Actions

EDA is only useful if it drives decisions. Every finding maps to a concrete next step.

| Finding | Action |
|---|---|
| SalePrice right-skewed | Predict log(SalePrice), exponentiate predictions |
| LotArea right-skewed | log1p(LotArea) as feature |
| GarageCars has missing | Impute with 0 (no garage), add `has_garage` flag |
| OverallQual accelerates exponentially | Add OverallQual^2 |
| Neighborhood huge spread | Target encode or ordinal encode by median price |
| GrLivArea * OverallQual synergy | Add interaction feature |
| YearBuilt non-linear | Use HouseAge, consider HouseAge^2 |
| TotRmsAbvGrd correlated with GrLivArea | Create area_per_room = GrLivArea/TotRmsAbvGrd |


In [ ]:
print("=" * 65)
print("  EDA SUMMARY REPORT — House Prices Dataset")
print("=" * 65)
print()

# 1. Missing values
print("1. MISSING VALUES")
miss = df.isnull().sum()
for col, n in miss[miss > 0].items():
    pct = n / len(df) * 100
    print(f"   {col:15}: {n:3d} missing ({pct:.1f}%)")
print()

# 2. Skewness report
print("2. SKEWED FEATURES (|skew| > 0.75)")
for col in NUMERIC_COLS:
    skew = df_clean[col].skew()
    if abs(skew) > 0.75:
        action = "log1p()" if skew > 0 else "check for floor effects"
        print(f"   {col:15}: skew={skew:+.2f}  -> {action}")
print()

# 3. Feature-target correlations
print("3. FEATURE-TARGET CORRELATIONS (|r| with SalePrice)")
corrs = df_clean[NUMERIC_COLS].corr()['SalePrice'].drop('SalePrice')
for feat, r in corrs.abs().sort_values(ascending=False).items():
    sign = '+' if corrs[feat] > 0 else '-'
    tier = "Strong" if abs(r) > 0.6 else ("Moderate" if abs(r) > 0.4 else "Weak")
    print(f"   {feat:15}: {corrs[feat]:+.3f}  ({tier})")
print()

# 4. Categorical signal strength
print("4. CATEGORICAL SIGNAL STRENGTH (price spread between categories)")
for cat in CAT_COLS:
    group_medians = df_clean.groupby(cat)['SalePrice'].median()
    spread = (group_medians.max() - group_medians.min()) / df_clean['SalePrice'].median() * 100
    print(f"   {cat:15}: {spread:.1f}% spread between highest/lowest category")
print()

# 5. Recommended actions
print("5. RECOMMENDED FEATURE ENGINEERING ACTIONS")
actions = [
    ("TARGET",     "Predict log1p(SalePrice), exponentiate predictions back"),
    ("LotArea",    "Use log1p(LotArea) to compress right tail"),
    ("GarageCars", "Impute missing with 0; add has_garage binary flag"),
    ("TotalBsmtSF","Impute missing with 0; add has_basement binary flag"),
    ("NEW",        "HouseAge = 2024 - YearBuilt (more intuitive than year)"),
    ("NEW",        "OverallQual^2 (price accelerates exponentially with quality)"),
    ("NEW",        "GrLivArea * OverallQual (size x quality interaction)"),
    ("NEW",        "GrLivArea / TotRmsAbvGrd (average room size)"),
    ("Neighborhood","Target encode: map each neighborhood to its mean log(SalePrice)"),
    ("Condition",  "Ordinal encode: Poor=1 ... Excellent=5"),
]
for col, action in actions:
    print(f"   {col:15}: {action}")
print()
print("=" * 65)
print("  These decisions directly drive feature_engineering.ipynb")
print("=" * 65)


---
## Summary — EDA Checklist

### The 5-Command Quick Start (run on every new dataset)

```python
print(df.shape)
print(df.dtypes)
print(df.describe())
print(df.isnull().sum())
print(df['target'].skew())
```

### The EDA Workflow

```
1. Univariate
   df[num_cols].hist(bins=30, figsize=(14,10))
   df[num_cols].skew().sort_values(ascending=False)

2. Target
   plt.hist(df['target'], bins=40)
   scipy.stats.probplot(df['target'], plot=ax)   # Q-Q plot

3. Bivariate (numeric)
   df[num_cols].corr()['target'].sort_values()
   scatter: plt.scatter(df['feat'], np.log1p(df['target']))

4. Bivariate (categorical)
   df.groupby('cat_col')['target'].agg(['mean','median','count'])
   sns.violinplot(data=df, x='cat_col', y='target')

5. Multicollinearity
   corr_matrix = df[num_cols].corr()
   sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r')

6. Pairplot (top 5 features)
   sns.pairplot(df[top5 + ['target']], hue='category')
```

### EDA -> Engineering Decision Map

| Finding | Action |
|---|---|
| Right skew > 1 | log1p() transform |
| Many zeros + tail | binary flag + log1p |
| Categorical spread | target encode or ordinal |
| High cardinality | target encode or frequency encode |
| Feature-feature correlation > 0.8 | drop one, or create ratio |
| Non-linear vs target | add squared term or bin the feature |
| Strong visual interaction | multiply the two features |

### Next Notebook
`feature_engineering.ipynb` — build every feature the EDA told you to build
